# Fluor Unified: Fluorescent Molecule Property Prediction

In [ ]:
import subprocess, sys, os
import torch

# Detect CUDA version for DGL wheel selection
cuda_version = None
if torch.cuda.is_available():
    cuda_version = torch.version.cuda
    print(f"CUDA {cuda_version} detected")
else:
    print("No CUDA detected, using CPU")

# Install DGL from the correct wheel URL
# DGL 2.4 supports up to PyTorch 2.4 (ref: https://github.com/dmlc/dgl/issues/7822)
torch_ver = ".".join(torch.__version__.split(".")[:2])
if cuda_version:
    dgl_url = f"https://data.dgl.ai/wheels/torch-{torch_ver}/cu{cuda_version.replace('.','')}/repo.html"
else:
    dgl_url = f"https://data.dgl.ai/wheels/torch-{torch_ver}/repo.html"

subprocess.run([sys.executable, "-m", "pip", "install", "dgl", "-f", dgl_url], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "rdkit", "dgllife", "tqdm"], check=False)

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import torch._dynamo
torch._dynamo.config.suppress_errors = True
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO_URL = "https://github.com/your-org/fluor-tools.git"  # placeholder
REPO_DIR = "/content/fluor-tools"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

# Add to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Set up data directories
DRIVE_ROOT = "/content/drive/MyDrive/fluor-tools"
DATASETS_DIR = f"{DRIVE_ROOT}/datasets"
ACTIVE_RUN_DIR = f"{DRIVE_ROOT}/training-runs/active"
COMPLETED_RUNS_DIR = f"{DRIVE_ROOT}/training-runs/completed"
for d in [DATASETS_DIR, ACTIVE_RUN_DIR, COMPLETED_RUNS_DIR]:
    os.makedirs(d, exist_ok=True)
print("Setup complete")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

mode_selector = widgets.RadioButtons(
    options=["Create Training Data", "Predict Properties", "Train Models"],
    description="Mode:",
    layout=widgets.Layout(width="400px"),
)
display(mode_selector)

In [ ]:
# Run this cell when mode == 'Create Training Data'
import io
from pathlib import Path

import pandas as pd

from colab_training.fluor_modules import data_pipeline

UPLOAD_DIR = f'{DATASETS_DIR}/uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

STANDARD_INPUT_COLUMNS = [
    'name', 'solvent', 'abs', 'em', 'epsilon', 'mw', 'plqy', 'smiles',
]
EXCEL_COLUMN_MAP = {
    'Name/ Nummer': 'name',
    'LM': 'solvent',
    'Abs': 'abs',
    'Flu': 'em',
    'Epsilon L/(cm*mol)': 'epsilon',
    'MW': 'mw',
    'QY': 'plqy',
    'SMILES': 'smiles',
}

def _uploaded_file_to_name_and_bytes(upload_widget):
    value = upload_widget.value
    if not value:
        return None, None

    if isinstance(value, dict):
        first_key = next(iter(value))
        upload = value[first_key]
    else:
        upload = value[0]
        first_key = None

    if hasattr(upload, 'keys'):
        name = upload.get('name') or first_key or 'uploaded_file'
        content = upload.get('content')
    else:
        name = getattr(upload, 'name', first_key) or 'uploaded_file'
        content = getattr(upload, 'content', None)

    if isinstance(content, memoryview):
        content = content.tobytes()
    elif isinstance(content, bytearray):
        content = bytes(content)

    return name, content

def _normalize_training_input_df(df):
    if set(STANDARD_INPUT_COLUMNS).issubset(df.columns):
        normalized_df = df[STANDARD_INPUT_COLUMNS].copy()
        detected_format = 'standard CSV'
    elif set(EXCEL_COLUMN_MAP).issubset(df.columns):
        normalized_df = df[list(EXCEL_COLUMN_MAP)].rename(columns=EXCEL_COLUMN_MAP)
        detected_format = 'Excel template'
    else:
        raise ValueError(
            'Unsupported columns. Provide a CSV with columns '
            f'{STANDARD_INPUT_COLUMNS} or an Excel file with columns '
            f'{list(EXCEL_COLUMN_MAP)}.'
        )

    original_rows = len(normalized_df)
    normalized_df = normalized_df.dropna(subset=['smiles']).copy()
    return normalized_df, detected_format, original_rows

def prepare_training_input_csv(file_name, file_bytes=None, input_path=None):
    source_name = file_name or Path(input_path).name
    suffix = Path(source_name).suffix.lower()

    if file_bytes is not None:
        buffer = io.BytesIO(file_bytes)
        if suffix == '.csv':
            df = pd.read_csv(buffer)
        elif suffix in {'.xlsx', '.xls'}:
            df = pd.read_excel(buffer)
        else:
            raise ValueError('Only .csv, .xlsx, and .xls files are supported.')
    else:
        if suffix == '.csv':
            df = pd.read_csv(input_path)
        elif suffix in {'.xlsx', '.xls'}:
            df = pd.read_excel(input_path)
        else:
            raise ValueError('Only .csv, .xlsx, and .xls files are supported.')

    normalized_df, detected_format, original_rows = _normalize_training_input_df(df)
    normalized_path = os.path.join(
        UPLOAD_DIR, f'{Path(source_name).stem}_normalized.csv'
    )
    normalized_df.to_csv(normalized_path, index=False)

    summary = {
        'source_name': source_name,
        'source_format': detected_format,
        'original_rows': original_rows,
        'rows_with_smiles': len(normalized_df),
        'normalized_csv_path': normalized_path,
    }
    return normalized_path, summary

data_file_upload = widgets.FileUpload(
    accept='.csv,.xlsx,.xls',
    multiple=False,
    description='Upload file',
)
data_path_input = widgets.Text(
    value='',
    placeholder='/content/drive/MyDrive/fluor-tools/datasets/my_data.xlsx',
    description='Or path:',
    layout=widgets.Layout(width='600px'),
)
display(widgets.HTML(
    '<b>Upload a .csv, .xlsx, or .xls file</b><br>'
    'Excel files are converted to the required CSV format automatically.'
))
display(data_file_upload)
display(data_path_input)

In [ ]:
import pandas as pd
from IPython.display import display

uploaded_name, uploaded_bytes = _uploaded_file_to_name_and_bytes(data_file_upload)
input_path = data_path_input.value.strip()

if uploaded_bytes is None and not input_path:
    print('Upload a file above or enter a path to a .csv, .xlsx, or .xls file.')
else:
    try:
        if uploaded_bytes is not None:
            csv_path, input_summary = prepare_training_input_csv(
                file_name=uploaded_name,
                file_bytes=uploaded_bytes,
            )
            if input_path:
                print('Using uploaded file and ignoring the path field.')
        else:
            csv_path, input_summary = prepare_training_input_csv(
                file_name=os.path.basename(input_path),
                input_path=input_path,
            )

        print(
            f"Prepared {input_summary['source_format']} input from "
            f"{input_summary['source_name']}"
        )
        print(
            f"Rows before SMILES cleanup: {input_summary['original_rows']}; "
            f"rows kept: {input_summary['rows_with_smiles']}"
        )
        print(f"Normalized CSV saved to: {input_summary['normalized_csv_path']}")

        solvent_mapping = data_pipeline.load_solvent_mapping(
            f'{REPO_DIR}/Fluor-RLAT/data/00_solvent_mapping.csv'
        )
        patterns = data_pipeline.load_substructure_patterns(
            f'{REPO_DIR}/Fluor-RLAT/data/00_mmp_substructure.csv'
        )
        main_df, smiles_fp_df, sol_fp_df, skipped = data_pipeline.process_input_csv(
            csv_path, solvent_mapping, patterns
        )
        print(f'Processed {len(main_df)} valid rows, {len(skipped)} skipped')
        if skipped:
            print('Skipped rows:')
            for name, reason in skipped:
                print(f'  {name}: {reason}')
        display(main_df.head())
    except Exception as exc:
        print(f'Failed to prepare/process the input file: {exc}')

In [ ]:
# Merge with existing training data and save to Google Drive
split_data = data_pipeline.split_by_property(main_df, smiles_fp_df, sol_fp_df)
for prop, (prop_main, prop_smiles, prop_sol) in split_data.items():
    existing_main_path = f"{DATASETS_DIR}/train_{prop}.csv"
    if os.path.exists(existing_main_path):
        existing_main = pd.read_csv(existing_main_path)
        existing_smiles = pd.read_csv(f"{DATASETS_DIR}/train_smiles_{prop}.csv")
        existing_sol = pd.read_csv(f"{DATASETS_DIR}/train_sol_{prop}.csv")
        merged_main, merged_smiles, merged_sol = data_pipeline.merge_training_data(
            prop_main, prop_smiles, prop_sol,
            existing_main, existing_smiles, existing_sol,
        )
        print(f"{prop}: merged {len(prop_main)} new + {len(existing_main)} existing = {len(merged_main)} rows")
    else:
        merged_main, merged_smiles, merged_sol = prop_main, prop_smiles, prop_sol
        print(f"{prop}: {len(merged_main)} new rows (no existing data)")
    merged_main.to_csv(f"{DATASETS_DIR}/train_{prop}.csv", index=False)
    merged_smiles.to_csv(f"{DATASETS_DIR}/train_smiles_{prop}.csv", index=False)
    merged_sol.to_csv(f"{DATASETS_DIR}/train_sol_{prop}.csv", index=False)
print("Saved to Google Drive")

In [ ]:
model_source = widgets.RadioButtons(
    options=["pretrained", "custom"],
    description="Model source:",
)
display(model_source)

In [ ]:
smiles_input = widgets.Textarea(
    value="",
    placeholder="Enter SMILES (one per line) or leave empty to use CSV",
    description="SMILES:",
    layout=widgets.Layout(width="600px", height="100px"),
)
solvent_input = widgets.Text(
    value="EtOH",
    description="Solvent:",
    layout=widgets.Layout(width="300px"),
)
display(smiles_input, solvent_input)

In [ ]:
from colab_training.fluor_modules import prediction_engine

# Resolve model directory
if model_source.value == "pretrained":
    model_dir = f"{REPO_DIR}/Fluor-RLAT"
else:
    model_dir = prediction_engine.find_latest_completed_run(COMPLETED_RUNS_DIR)
    if model_dir is None:
        print("No custom models found. Please train models first or use pretrained.")
        model_dir = None

if model_dir:
    data_dir = f"{REPO_DIR}/Fluor-RLAT/data"
    from fluor_modules.data_pipeline import map_solvent_name_to_smiles
    sol_smiles = map_solvent_name_to_smiles(solvent_input.value) or solvent_input.value

    smiles_list = [s.strip() for s in smiles_input.value.strip().split("\n") if s.strip()]
    if not smiles_list:
        print("Please enter at least one SMILES string")
    else:
        results = []
        for smi in smiles_list:
            preds = prediction_engine.predict_all_properties(smi, sol_smiles, model_dir, data_dir, device)
            preds["smiles"] = smi
            results.append(preds)
        results_df = pd.DataFrame(results)
        display(results_df)

In [ ]:
from colab_training.fluor_modules import training_engine
import uuid

prop_checkboxes = {
    p: widgets.Checkbox(value=True, description=p)
    for p in ["abs", "em", "plqy", "k"]
}
data_source_selector = widgets.RadioButtons(
    options=["original repo data", "merged Google Drive data"],
    description="Data source:",
)
display(*prop_checkboxes.values(), data_source_selector)

In [ ]:
selected_targets = [p for p, cb in prop_checkboxes.items() if cb.value]
if not selected_targets:
    print("Please select at least one property to train")
else:
    if data_source_selector.value == "merged Google Drive data":
        data_dir = DATASETS_DIR
    else:
        data_dir = f"{REPO_DIR}/Fluor-RLAT/data"

    from fluor_modules.config import MODEL_REGISTRY, LEARNING_RATE, BATCH_SIZE
    model_configs = {t: MODEL_REGISTRY[t] for t in selected_targets}
    run_config = training_engine.TrainingConfig(
        targets=selected_targets,
        epochs=200,
        patience=30,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        model_configs=model_configs,
        data_source=data_dir,
        lr_scheduler_factor=0.5,
        lr_scheduler_patience=10,
        lr_scheduler_min=1e-6,
        run_id=str(uuid.uuid4()),
    )

    session_status = training_engine.check_existing_session(ACTIVE_RUN_DIR, run_config)
    if session_status == "mismatch":
        print("WARNING: Existing session has different config. Starting fresh.")
    elif session_status == "match":
        print("Resuming existing session...")

    # Save config
    import json
    os.makedirs(ACTIVE_RUN_DIR, exist_ok=True)
    with open(f"{ACTIVE_RUN_DIR}/training_config.json", "w") as f:
        f.write(run_config.to_json())

    all_metrics = {}
    for target in selected_targets:
        print(f"\nTraining {target}...")
        metrics = training_engine.train_model(
            target=target,
            data_dir=data_dir,
            active_dir=ACTIVE_RUN_DIR,
            config=MODEL_REGISTRY[target],
            epochs=run_config.epochs,
            patience=run_config.patience,
            learning_rate=run_config.learning_rate,
            device=device,
        )
        all_metrics[target] = metrics
        print(f"{target}: MAE={metrics['mae']:.4f}, RMSE={metrics['rmse']:.4f}, R2={metrics['r2']:.4f}")

    archive_path = training_engine.archive_completed_run(ACTIVE_RUN_DIR, COMPLETED_RUNS_DIR, selected_targets)
    print(f"\nTraining complete. Models archived to: {archive_path}")
    display(pd.DataFrame(all_metrics).T[["mae", "rmse", "r2"]])